<a href="https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/04_structured_outputs/notebook_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4: Structured Outputs

This notebook explores **Structured Outputs** for guiding LLM outputs.

We will use the `google-genai` library to interact with Google's Gemini models.

**Learning Objectives:**

1.  **Understand structured outputs** and why they are crucial for reliable data extraction from LLMs.
2.  **Enforce structured data formats (JSON)** from an LLM using prompt engineering techniques.
3.  **Leverage Pydantic models** to define and manage complex data structures for structured outputs, improving code robustness and clarity.
4.  **Use Gemini's native structured output capabilities** for the most reliable and efficient approach.

> **Exercise version.** This is the exercise notebook for Lesson 4. The full solution lives in [`notebook.ipynb`](https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/04_structured_outputs/notebook.ipynb) in the same folder. Attempt each exercise before checking the solutions: the struggle is where the learning happens.

### Exercise roadmap

| Exercise | Difficulty | What you build |
|---|---|---|
| 1 | Intermediate | A prompt that forces the LLM to emit a specific JSON structure |
| 2 | Starter | A parser that extracts JSON from tagged LLM responses |
| 3 | Intermediate | A schema-injected prompt built from a Pydantic model |
| 4 | Intermediate | Native structured outputs with Gemini's `GenerateContentConfig` |

## 1. Setup


### Set Up Python Environment

**Google Colab:** Run the code cell below — it installs all required packages and automatically loads your credentials from Colab Secrets (your `GOOGLE_API_KEY`, or your Vertex AI settings if you chose that option in the Course Admin lesson).


To set up your Python virtual environment using `uv` and load it into the Notebook, follow the step-by-step instructions from the `Course Admin` lesson from the beginning of the course.

**TL;DR:** Be sure the correct kernel pointing to your `uv` virtual environment is selected.

In [ ]:
# ============================================================
# Google Colab Setup — runs only when executed in Colab
# ============================================================
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import importlib
    import os
    import site
    import subprocess

    # Install the course package (published from pyproject.toml) and its pinned extras
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "agentic-ai-engineering-course==0.4.8",
            "nest-asyncio2",
            "google-auth==2.53.0",
            "opentelemetry-api==1.42.1",
            "opentelemetry-sdk==1.42.1",
            "opentelemetry-exporter-otlp-proto-http==1.42.1",
            "opentelemetry-exporter-otlp-proto-common==1.42.1",
            "opentelemetry-proto==1.42.1",
            "jedi==0.18.2",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without restart

    # Load API key from Colab Secrets
    # In Colab: Secrets tab (key icon) → Add new secret → Name: GOOGLE_API_KEY
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    from utils import env

    env.load(required_env_vars=["GOOGLE_API_KEY"])

### Import Key Packages

In [ ]:
import json

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from utils import pretty_print

### Initialize the Gemini Client

In [ ]:
client = genai.Client()

### Define Constants

We will use the `gemini-3.5-flash` model, which is fast and cost-effective:

In [ ]:
MODEL_ID = "gemini-3.5-flash"

## 2. Implementing structured outputs from scratch using JSON

Sometimes, you don't need the LLM to take an action, but you need its output in a specific, machine-readable format. Forcing the output to be JSON is a common way to achieve this.

We can instruct the model to do this by **prompting** clearly describing the desired JSON structure in the prompt.

### Example: Extracting Metadata from a Document

Let's imagine we have a markdown document and we want to extract key information like a summary, tags, and keywords into a clean JSON object.

### Exercise 1: Prompt the model into emitting JSON

The first way to get structured data out of an LLM is plain prompt engineering: describe the exact JSON shape you want inside the prompt, then parse what comes back.

**Learning goal:** Enforce a JSON output format from an LLM using only the prompt.

**What you need to implement:**

1. Build a prompt that instructs the model to analyze `DOCUMENT` and return a single valid JSON object
2. Describe the expected structure inside the prompt (wrap the example structure in `<json>` tags) with these fields: `summary`, `tags`, `keywords`, `quarter`, `growth_rate`
3. Include the document itself in the prompt (wrap it in `<document>` tags)
4. Call the model and print the raw output

**Key concepts:**

- f-strings with doubled braces (`{{` and `}}`) let you write literal JSON braces inside an f-string
- `client.models.generate_content()` is the basic Gemini text call, it takes `model` and `contents`
- XML-style tags like `<json>` and `<document>` help the model separate instructions from data

**Expected output:** the raw model text, printed via `pretty_print.wrapped`, containing a JSON object with the five fields above (it may be wrapped in `<json>` or ```` ```json ```` fences, that is what Exercise 2 cleans up).

**Implementation hints:**

- Show the model an example of the structure, field names plus a short description of each value
- Note: the solved parsing cell a bit further down consumes `response`, it will error until you complete this exercise

In [ ]:
# === Exercise cell: fill in the gaps below ===

# Steps to complete:
# 1. Write a prompt that asks the model to extract metadata from DOCUMENT
#    and to answer with a single valid JSON object only
# 2. Describe the expected JSON structure inside <json> tags in the prompt
#    (fields: summary, tags, keywords, quarter, growth_rate)
# 3. Embed the document inside <document> tags
# 4. Call the model with client.models.generate_content and store the result in `response`

DOCUMENT = """
# Q3 2023 Financial Performance Analysis

The Q3 earnings report shows a 20% increase in revenue and a 15% growth in user engagement,
beating market expectations. These impressive results reflect our successful product strategy
and strong market positioning.

Our core business segments demonstrated remarkable resilience, with digital services leading
the growth at 25% year-over-year. The expansion into new markets has proven particularly
successful, contributing to 30% of the total revenue increase.

Customer acquisition costs decreased by 10% while retention rates improved to 92%,
marking our best performance to date. These metrics, combined with our healthy cash flow
position, provide a strong foundation for continued growth into Q4 and beyond.
"""

prompt = None  # TODO 1: build the JSON-forcing prompt described above
response = None  # TODO 2: call the model with your prompt

# Your implementation goes here

# Smoke test: prints the raw output once the exercise is complete
if response is not None:
    pretty_print.wrapped(text=response.text, title="Raw LLM Output", indent=2)
else:
    print("(response not set yet, complete the TODOs above)")

### Exercise 2: Parse JSON out of a raw LLM response

Models often wrap JSON in `<json>` tags or markdown code fences even when told not to. Before `json.loads` can run, those wrappers have to go.

**Learning goal:** Turn a raw LLM response string into a Python dictionary reliably.

**What you need to implement:**

1. Remove `<json>` and `</json>` tags from the response string
2. Remove markdown code fences (the ```` ```json ```` opener and plain ```` ``` ```` closer)
3. Parse the cleaned string and return the resulting dictionary

**Key concepts:**

- `str.replace(old, new)` returns a new string with occurrences swapped out
- `json.loads()` parses a JSON string into Python objects and raises if the string is not valid JSON

**Expected output:** the smoke test below the function prints a parsed `dict`, for example `{'status': 'ok'}`.

**Implementation hints:**

- Chain several `replace` calls, each one strips a different wrapper
- The solved cell further down reuses this function on the real `response` from Exercise 1

In [ ]:
# === Exercise cell: fill in the gaps below ===


def extract_json_from_response(response: str) -> dict:
    """
    Extracts JSON from a response string that is wrapped in <json> or ```json tags.

    Steps to complete:
    1. Strip the <json> and </json> tags from the string
    2. Strip the ```json and ``` markdown fences from the string
    3. Parse the cleaned string with json.loads and return the result
    """

    return {}  # Replace with the real parsed dictionary


# Smoke test (no API call): should print {'status': 'ok'}
_test = extract_json_from_response('<json>{"status": "ok"}</json>')
print(_test if _test else "(empty dict, complete the function above)")

You can now reliably parse the JSON string:

In [ ]:
parsed_response = extract_json_from_response(response.text)
pretty_print.wrapped(
    text=[f"Type of the parsed response: `{type(parsed_response)}`", json.dumps(parsed_response, indent=2)],
    title="Parsed JSON Object",
    indent=2,
)

## 3. Implementing structured outputs from scratch using Pydantic

While prompting for JSON is effective, it can be fragile. A more robust and modern approach is to use **Pydantic**. Pydantic allows you to define data structures as Python classes. This gives you:

- **A single source of truth**: The Pydantic model defines the structure.
- **Automatic schema generation**: You can easily generate a JSON Schema from the model.
- **Data validation**: You can validate the LLM's output against the model to ensure it conforms to the expected structure and types.

Let's recreate the previous example using Pydantic.

In [ ]:
class DocumentMetadata(BaseModel):
    """A class to hold structured metadata for a document."""

    summary: str = Field(description="A concise, 1-2 sentence summary of the document.")
    tags: list[str] = Field(description="A list of 3-5 high-level tags relevant to the document.")
    keywords: list[str] = Field(description="A list of specific keywords or concepts mentioned.")
    quarter: str = Field(description="The quarter of the financial year described in the document (e.g, Q3 2023).")
    growth_rate: str = Field(description="The growth rate of the company described in the document (e.g, 10%).")

### Injecting Pydantic Schema into the Prompt

We can generate a JSON Schema from our Pydantic model and inject it directly into the prompt. This is a more formal way of telling the LLM what structure to follow.

Note how, along with the field type, we can leverage the Field description automatically to clearly specify to the LLM what each field means.

In [ ]:
schema = DocumentMetadata.model_json_schema()
schema

### Exercise 3: Inject a Pydantic schema into the prompt

Hand-writing the JSON structure in every prompt is repetitive and drifts out of sync. A Pydantic model can be the single source of truth: generate its JSON Schema and inject that into the prompt instead.

**Learning goal:** Drive the output format from a Pydantic model by embedding its generated JSON Schema in the prompt.

**What you need to implement:**

1. Build a prompt that embeds the `schema` generated in the previous cell (serialize it with `json.dumps`, wrap it in `<json>` tags) and the document (in `<document>` tags)
2. Call the model with that prompt
3. Parse the response text with your `extract_json_from_response` from Exercise 2
4. Print the parsed result

**Key concepts:**

- `DocumentMetadata.model_json_schema()` produces a JSON Schema dict, including each `Field` description
- `json.dumps(schema, indent=2)` renders the schema readably inside the prompt

**Expected output:** a parsed `dict` with the same five keys as Exercise 1, printed via `pretty_print.wrapped` with the title "Parsed JSON Object".

**Implementation hints:**

- The prompt shape mirrors Exercise 1, the only change is what goes inside the `<json>` tags
- The solved validation cell below feeds `parsed_response` into `DocumentMetadata.model_validate`, it prints a validation failure until this exercise is complete

In [ ]:
# === Exercise cell: fill in the gaps below ===

# Steps to complete:
# 1. Build a prompt that embeds the JSON Schema from `schema` (as a json.dumps string
#    inside <json> tags) and the document (inside <document> tags)
# 2. Call the model and store the result in `response`
# 3. Parse response.text with extract_json_from_response into `parsed_response`

prompt = None  # TODO 1: build the schema-injected prompt
response = None  # TODO 2: call the model
parsed_response = {}  # TODO 3: parse the response text

# Your implementation goes here

# Smoke test: prints the parsed object once the exercise is complete
if parsed_response:
    pretty_print.wrapped(
        text=[f"Type of the parsed response: `{type(parsed_response)}`", json.dumps(parsed_response, indent=2)],
        title="Parsed JSON Object",
        indent=2,
    )
else:
    print("(parsed_response is empty, complete the TODOs above)")

As you can see, conceptually, the results are the same. But now, we can easily validate the output with Pydantic:

In [ ]:
try:
    document_metadata = DocumentMetadata.model_validate(parsed_response)
    print("\nValidation successful!")

    pretty_print.wrapped(
        ["Type of the validated response: `{type(document_metadata)}`", document_metadata.model_dump_json(indent=2)],
        title="Pydantic Validated Object",
        indent=2,
    )
except Exception as e:
    print(f"\nValidation failed: {e}")

## 4. Implementing structured outputs using Gemini and Pydantic

Using Gemini's `GenerateContentConfig` we can enforce the output as a Pydantic object without any special prompt engineering.

We can instruct the model to do this by setting `response_mime_type` to `"application/json"` in the generation configuration, which forces the model's output to be a valid JSON object and the `response_schema` to our Pydantic object.

**Note:** If you use only the `response_mime_type="application/json"` setting you can output raw JSON formats.

### Exercise 4: Use Gemini's native structured outputs

This is the approach the rest of the course relies on: no schema text in the prompt at all. The API itself constrains generation, and the SDK hands you back a validated Pydantic object.

**Learning goal:** Configure Gemini to return a validated `DocumentMetadata` instance directly.

**What you need to implement:**

1. Build a `types.GenerateContentConfig` that forces JSON output (set the response MIME type to JSON) and validates against the `DocumentMetadata` schema
2. Write a short prompt that asks for the document's metadata, no format instructions needed
3. Call the model, passing the config alongside model and contents
4. Print `response.parsed`

**Key concepts:**

- `types.GenerateContentConfig` carries generation settings, including `response_mime_type` and `response_schema`
- `response.parsed` is the SDK-validated Pydantic instance, no manual parsing step

**Expected output:** `pretty_print.wrapped` output titled "Pydantic Validated Object" showing type `DocumentMetadata` and its five fields as JSON.

**Implementation hints:**

- Compare with Exercise 3: the prompt shrinks because the structure moved into the config
- Uncomment and run the validation cell below once you are done

In [ ]:
# === Exercise cell: fill in the gaps below ===

# Steps to complete:
# 1. Build a GenerateContentConfig that forces JSON output and validates it
#    against the DocumentMetadata schema
# 2. Write a short prompt asking the model to extract the document's metadata
#    (embed DOCUMENT inside <document> tags, no schema text needed)
# 3. Call the model with the prompt AND the config, store the result in `response`

config = None  # TODO 1: build the generation config
prompt = None  # TODO 2: write the prompt
response = None  # TODO 3: call the model with the config

# Your implementation goes here


# Smoke test: prints the validated object once the exercise is complete
if response is not None and getattr(response, "parsed", None) is not None:
    pretty_print.wrapped(
        [f"Type of the response: `{type(response.parsed)}`", response.parsed.model_dump_json(indent=2)],
        title="Pydantic Validated Object",
        indent=2,
    )
else:
    print("(response not set yet, complete the TODOs above)")

### Validation check - run this after your implementation

Uncomment the cell below and run it once Exercise 4 is implemented. It checks that the response carries a validated `DocumentMetadata` object.

In [ ]:
# # Validation: check your Exercise 4 implementation
# try:
#     assert response is not None, "❌ 'response' is not set. Run your Exercise 4 cell first."
#     assert config is not None, "❌ 'config' is not set. Build the GenerateContentConfig first."
#     assert getattr(response, "parsed", None) is not None, "❌ response.parsed is None. Did you pass the config to the model call?"
#     assert isinstance(response.parsed, DocumentMetadata), "❌ Expected response.parsed to be a DocumentMetadata instance."
#     assert response.parsed.summary, "❌ The summary field is empty."
#     assert len(response.parsed.tags) >= 1, "❌ Expected at least one tag."
#     print("✅ All checks passed! response.parsed is a validated DocumentMetadata object.")
# except AssertionError as e:
#     print(e)
#     print("💡 Tip: the config needs both a JSON response MIME type and the DocumentMetadata schema.")

From now on, throughout this course, we will utilize this native Gemini approach to generate structured outputs, aiming to achieve the most reliable and efficient results. Additionally, when using LangChain or LangGraph, we will utilize their abstractions on top of the same logic.

## Stretch challenges

Want to go further? Try these on your own:

1. Extend `DocumentMetadata` with a `people_mentioned` field (name, role, sentiment for each person) and re-run the native extraction from Exercise 4.
2. Add a `risk_factors: list[str]` field, then compare extraction quality with and without a few-shot example in the prompt.
3. Wrap the Exercise 3 flow in a retry loop: if `DocumentMetadata.model_validate` raises, re-prompt the model with the validation error message appended.